# 10. Reproducibilidad y bibliografía

Este capítulo reúne lo necesario para reconstruir los resultados: el entorno (software y hardware, leídos en el momento de ejecutar el
libro), las semillas, los rangos de hiperparámetros, el número de ejecuciones y su dispersión, la tabla maestra de experimentos, el
tiempo de cómputo, los enlaces y la lista de verificación de la guía (sec. 9.2).

In [1]:
from pathlib import Path
import importlib.metadata as im
import json
import os
import platform
import subprocess
import sys

import numpy as np
import pandas as pd

RAIZ = Path.cwd()                       # raiz del repositorio (sin rutas absolutas)
assert (RAIZ / "experiments" / "protocolo.md").is_file(), "Ejecute este cuaderno desde la raiz del repositorio"
sys.path.insert(0, str(RAIZ)); sys.path.insert(0, str(RAIZ / "experiments"))
os.environ.setdefault("E4_RAIZ_PROYECTO", str(RAIZ))
pd.set_option("display.max_colwidth", 120)

## 10.1 Entorno de software y hardware

In [2]:
paquetes = ["numpy", "scipy", "pandas", "scikit-learn", "xgboost", "lightgbm", "catboost", "optuna", "deap", "imbalanced-learn",
            "faiss-cpu", "shap", "statsmodels", "matplotlib", "pyarrow", "joblib", "nbclient", "jupyter-book", "pytest"]
ver = {}
for p in paquetes:
    try:
        ver[p] = im.version(p)
    except im.PackageNotFoundError:
        ver[p] = "no instalado"
try:
    import psutil
    ram = f"{psutil.virtual_memory().total / 2**30:.1f} GB"
except ImportError:
    ram = "desconocida"
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=20).stdout.strip() or "sin GPU NVIDIA visible"
except Exception:
    gpu = "sin GPU NVIDIA visible"
entorno = pd.Series({"Python": platform.python_version(), "Sistema": f"{platform.system()} {platform.release()}",
                     "Procesador": platform.processor() or platform.machine(), "Nucleos logicos": os.cpu_count(),
                     "RAM": ram, "GPU": gpu, **ver}, name="version")
entorno.to_frame()

,version
Python,3.12.0
Sistema,Windows 11
Procesador,"Intel64 Family 6 Model 186 Stepping 2, GenuineIntel"
Nucleos logicos,12
RAM,15.6 GB
GPU,"NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144 MiB"
numpy,2.2.6
scipy,1.15.3
pandas,2.3.3
scikit-learn,1.7.2


**Lectura de la tabla 10.1.** Son las versiones con las que se ejecutó este libro. Los experimentos del E4 corrieron en el mismo
portátil: CPU Intel y GPU NVIDIA RTX 3050, con la GPU solo para XGBoost, LightGBM y CatBoost. `requirements.txt` fija las versiones
del E3 y `requirements_e4.txt` las del E4. Cada artefacto guarda además en `artifacts/metadata.json` las versiones con las que se ajustó.

## 10.2 Semillas

In [3]:
from src import config
import config_e4 as cfg
semillas = pd.DataFrame([
    ("Global (numpy, scikit-learn, XGBoost, Optuna, DEAP, FAISS)", config.SEMILLA, "src/config.py: SEMILLA y fijar_semillas()"),
    ("Particion desarrollo / prueba (80/20 estratificada)", config.SEMILLA, "cap. 01 del E3"),
    ("Pliegues externos (5) e internos (3) del E3 y de Home Credit en el E4", config.SEMILLA, "src/experimento.py"),
    ("ELAR-R y benchmarks del E4", cfg.SEMILLA, "experiments/config_e4.py: SEMILLA"),
    ("Conjuntos adicionales: repeticiones 5 x 2", "0, 1, 2, 3, 4", "experiments/e13_datasets_adicionales.py"),
    ("Estabilidad (modelos estocasticos)", ", ".join(map(str, cfg.SEMILLAS_ESTABILIDAD)), "experiments/e14_robustez.py"),
    ("Bootstrap BCa (2,000 replicas)", config.SEMILLA, "experiments/comun.py"),
], columns=["uso", "semilla", "donde"]).set_index("uso")
semillas

,semilla,donde
uso,,
"Global (numpy, scikit-learn, XGBoost, Optuna, DEAP, FAISS)",42,src/config.py: SEMILLA y fijar_semillas()
Particion desarrollo / prueba (80/20 estratificada),42,cap. 01 del E3
Pliegues externos (5) e internos (3) del E3 y de Home Credit en el E4,42,src/experimento.py
ELAR-R y benchmarks del E4,42,experiments/config_e4.py: SEMILLA
Conjuntos adicionales: repeticiones 5 x 2,"0, 1, 2, 3, 4",experiments/e13_datasets_adicionales.py
Estabilidad (modelos estocasticos),"0, 1, 2, 3, 4, 5, 6, 7, 8, 9",experiments/e14_robustez.py
"Bootstrap BCa (2,000 replicas)",42,experiments/comun.py


## 10.3 Rangos de hiperparámetros

Los espacios de búsqueda del E3 (las 140 combinaciones) están en `src/espacios.py`; los del E4 (búsqueda TPE en los conjuntos adicionales)
en `experiments/config_e4.py`. La configuración congelada de ELAR-R en Home Credit es la 7 (`BASE_ELAR` + `pares_cociente`).

In [4]:
from src import espacios
t_e3 = espacios.tabla_espacios()
t_e3 = t_e3[t_e3["Tarea"] == "clasificacion"].drop(columns="Tarea")
filas = []
for modelo, esp in {"elar (ELAR-R)": cfg.ESPACIO_ELAR, **cfg.ESPACIOS_BENCH}.items():
    for h, (tipo, a, b, log) in esp.items():
        filas.append({"Modelo": modelo, "Hiperparametro": h,
                      "Rango": ", ".join(map(str, a)) if tipo == "cat" else f"[{a:g}, {b:g}]",
                      "Escala": "categorica" if tipo == "cat" else ("log" if log else "lineal")})
t_e4 = pd.DataFrame(filas)
print(f"E3: {len(t_e3)} hiperparametros en 7 modelos de clasificacion | E4 (TPE, conjuntos adicionales): {len(t_e4)} en 10 modelos")
print("Configuracion fija de ELAR-R:", {k: v for k, v in cfg.BASE_ELAR.items() if k not in ("coste_fn", "coste_fp")})
t_e4

E3: 24 hiperparametros en 7 modelos de clasificacion | E4 (TPE, conjuntos adicionales): 40 en 10 modelos
Configuracion fija de ELAR-R: {'componentes': ('lineal', 'pares', 'profundo'), 'n_pliegues': 3, 'lam': 'auto', 'n_lambdas': 13, 'n_bins': 64, 'n_rondas_profundo': 2000, 'paciencia': 50, 'frac_validacion': 0.1, 'criterio_umbral': 'coste', 'max_iter_lineal': 500, 'pares_cociente': 20}


,Modelo,Hiperparametro,Rango,Escala
0,elar (ELAR-R),tasa_profundo,"[0.03, 0.3]",log
1,elar (ELAR-R),profundidad,"[3, 8]",lineal
2,elar (ELAR-R),frac_columnas_profundo,"[0.2, 0.8]",lineal
3,elar (ELAR-R),frac_filas,"[0.5, 1]",lineal
4,elar (ELAR-R),min_muestras_hoja,"[20, 1000]",log
5,elar (ELAR-R),lam_hoja,"[0.001, 100]",log
6,elar (ELAR-R),alpha_l1,"[1e-05, 0.01]",log
7,elar (ELAR-R),n_rondas_pares,"[50, 300]",lineal
8,elar (ELAR-R),tasa_aprendizaje,"[0.05, 0.5]",log
9,logistica,C,"[0.0001, 10]",log


In [5]:
t_e3.set_index(["Modelo", "Hiperparametro"])

Rango      Escala  \
Modelo        Hiperparametro                                       
knn           n_vecinos                     [5, 300]         log   
              ponderacion          uniform, distance  categorica   
              n_componentes                 [10, 80]      lineal   
naive_bayes   var_smoothing             [1e-12, 0.1]         log   
logistica     C                         [0.0001, 10]         log   
              penalizacion                    l1, l2  categorica   
arbol         max_depth                      [3, 20]      lineal   
              min_samples_leaf            [10, 2000]         log   
              max_features            sqrt, 0.3, 1.0  categorica   
random_forest n_estimators                [100, 300]      lineal   
              max_depth                      [6, 16]      lineal   
              min_samples_leaf            [20, 1000]         log   
              max_features               [0.02, 0.3]      lineal   
              max_samples                 [0.3, 0.7]      lineal   
xgboost       learning_rate              [0.01, 0.3]         log   
              max_depth                      [3, 10]      lineal   
              min_child_weight              [1, 100]         log   
              subsample                     [0.5, 1]      lineal   
              colsample_bytree              [0.2, 1]      lineal   
              reg_lambda                [0.001, 100]         log   
              reg_alpha                  [0.001, 10]         log   
svm           C                         [0.0001, 10]         log   
              nucleo            lineal, rbf_nystroem  categorica   
              gamma                   [0.0001, 0.01]         log   

                                                  Activo si  
Modelo        Hiperparametro                                 
knn           n_vecinos                             siempre  
              ponderacion                           siempre  
              n_componentes                         siempre  
naive_bayes   var_smoothing                         siempre  
logistica     C                                     siempre  
              penalizacion                          siempre  
arbol         max_depth                             siempre  
              min_samples_leaf                      siempre  
              max_features                          siempre  
random_forest n_estimators                          siempre  
              max_depth                             siempre  
              min_samples_leaf                      siempre  
              max_features                          siempre  
              max_samples                           siempre  
xgboost       learning_rate                         siempre  
              max_depth                             siempre  
              min_child_weight                      siempre  
              subsample                             siempre  
              colsample_bytree                      siempre  
              reg_lambda                            siempre  
              reg_alpha                             siempre  
svm           C                                     siempre  
              nucleo                                siempre  
              gamma             nucleo en ('rbf_nystroem',)

## 10.4 Tabla maestra de experimentos

In [6]:
tm = pd.read_csv(RAIZ / "resultados_e4" / "tabla_maestra_e4.csv", low_memory=False)
tm["fuente"] = tm["fuente"].fillna("E3 (140 combinaciones)")
resumen = tm.groupby("fuente").agg(filas=("modelo", "size"), modelos=("modelo", "nunique"),
                                   conjuntos=("conjunto", lambda s: ", ".join(sorted(s.dropna().astype(str).unique())) or "home_credit"))
print(f"resultados_e4/tabla_maestra_e4.csv (+ .parquet): {tm.shape[0]} filas x {tm.shape[1]} columnas")
print("Columnas:", ", ".join(tm.columns))
resumen

resultados_e4/tabla_maestra_e4.csv (+ .parquet): 500 filas x 49 columnas
Columnas: fuente, tarea, modelo, balanceo, optimizador, m_auc_roc_media, m_auc_pr_media, m_exactitud_media, m_precision_media, m_recall_media, m_especificidad_media, m_f1_media, m_umbral_media, m_brier_media, m_ece_media, m_ece_cuantiles_media, m_rmse_media, m_mae_media, m_r2_media, m_auc_roc_sd, m_auc_pr_sd, m_exactitud_sd, m_precision_sd, m_recall_sd, m_especificidad_sd, m_f1_sd, m_umbral_sd, m_brier_sd, m_ece_sd, m_ece_cuantiles_sd, m_rmse_sd, m_mae_sd, m_r2_sd, pliegues, auc_interno_media, evaluaciones_total, t_total_h, t_por_evaluacion_s, t_inferencia_s, semilla, params_por_pliegue, conjunto, pliegue, auc, repeticion, n_tr, n_va, t_total_s, estado


,filas,modelos,conjuntos
fuente,,,
E3 (140 combinaciones),140,10,home_credit
"E4 conjuntos adicionales (5x2, TPE 20, 3 internos)",300,10,"bank_marketing, german_credit, taiwan_default"
E4 igualdad (528 entradas),60,10,home_credit


**Lectura de la tabla 10.4.** La tabla maestra reúne en un solo archivo lo que pide la guía (sec. 10, ítem 6):

- las 140 combinaciones del E3, con métricas del bucle externo (media y desviación de los 5 pliegues), tiempos, número de evaluaciones
  y semilla;
- los 10 modelos en igualdad de condiciones en Home Credit (5 pliegues y la prueba);
- las 300 mediciones de los conjuntos adicionales: 10 modelos × 3 conjuntos × 10 particiones.

El registro por pliegue del E3 está en `resultados/registro_pliegues.csv`.

## 10.5 Número de ejecuciones, dispersión y tiempo de cómputo

In [7]:
R = pd.read_csv(RAIZ / "resultados_e4" / "f6_robustez" / "reentreno.csv").drop_duplicates(["estudio", "modelo", "variable", "valor"])
e4 = tm[tm["fuente"] != "E3 (140 combinaciones)"]
corridas = pd.DataFrame([
    ("E3: 140 combinaciones x 5 pliegues externos", 700, "media y desviacion de 5 pliegues", 444.5),
    ("E4: 10 modelos x 5 pliegues + prueba (Home Credit)", int((e4["conjunto"] == "home_credit").sum()), "desviacion de 5 pliegues; IC BCa", np.nan),
    ("E4: 10 modelos x 3 conjuntos x 10 particiones", int((e4["conjunto"] != "home_credit").sum()), "desviacion de 10 particiones; IC BCa",
     e4.loc[e4["conjunto"] != "home_credit", "t_total_s"].sum() / 3600),
    ("E4: reentrenos de robustez (pliegue 0, 60k filas)", len(R), "10 semillas en modelos estocasticos", R["segundos"].sum() / 3600),
], columns=["experimento", "ejecuciones", "dispersion reportada", "horas de computo"]).set_index("experimento")
corridas.round(1)

,ejecuciones,dispersion reportada,horas de computo
experimento,,,
E3: 140 combinaciones x 5 pliegues externos,700,media y desviacion de 5 pliegues,444.5
E4: 10 modelos x 5 pliegues + prueba (Home Credit),60,desviacion de 5 pliegues; IC BCa,NaN
E4: 10 modelos x 3 conjuntos x 10 particiones,300,desviacion de 10 particiones; IC BCa,24.1
"E4: reentrenos de robustez (pliegue 0, 60k filas)",279,10 semillas en modelos estocasticos,14.5


**Lectura de la tabla 10.5.** El E3 consumió 444.5 horas de cómputo (8.2 días de reloj con varias ventanas; anexo A1). En el E4, los
conjuntos adicionales y los reentrenos de robustez suman la mayor parte del tiempo. La comparación en igualdad en Home Credit no tiene
búsqueda (presupuesto cero), y su costo es el de un ajuste por pliegue de cada modelo (scripts `e10_igualdad.py` y `e9_ingenieria_variables.py`).

## 10.6 Lista de verificación de reproducibilidad

In [8]:
def existe(*rutas):
    return all((RAIZ / r).exists() for r in rutas)

n_pruebas = sum(open(p, encoding="utf-8").read().count("\ndef test_") for p in (RAIZ / "tests").glob("test_*.py"))
lista = pd.DataFrame([
    ("Versiones del software", "tabla 10.1; requirements.txt; requirements_e4.txt", existe("requirements.txt")),
    ("Hardware", "tabla 10.1 y sec. 10.1", True),
    ("Semillas", "tabla 10.2", True),
    ("Rangos de hiperparametros", "tablas 10.3 y 10.3b; src/espacios.py; experiments/config_e4.py", existe("src/espacios.py", "experiments/config_e4.py")),
    ("Numero de ejecuciones y dispersion", "tabla 10.5; tabla maestra", True),
    ("Tabla maestra (140 comb. + original + conjuntos adicionales)", "resultados_e4/tabla_maestra_e4.csv|parquet", existe("resultados_e4/tabla_maestra_e4.csv")),
    ("Protocolo pre-registrado con tag de Git", "experiments/protocolo.md; tags protocolo-e4 y protocolo-v2-evaluacion-ampliada", existe("experiments/protocolo.md")),
    ("Codigo que genera cada resultado", "experiments/ (scripts e9-e15) y CORRER_*.bat", existe("experiments/e15_estadistica_final.py")),
    ("Pruebas unitarias en integracion continua", f".github/workflows/pytest.yml; {n_pruebas} pruebas en tests/", existe(".github/workflows/pytest.yml")),
    ("Artefactos con metadata", "artifacts/*.joblib + metadata.json", existe("artifacts/metadata.json")),
    ("Ficha del modelo y declaracion de datos", "MODEL_CARD.md; DATA_STATEMENT.md", existe("MODEL_CARD.md", "DATA_STATEMENT.md")),
    ("Licencia y forma de citar", "LICENSE; CITATION.cff", existe("LICENSE", "CITATION.cff")),
    ("Enlace a los datos", "https://www.kaggle.com/competitions/home-credit-default-risk (no se redistribuyen)", True),
    ("Enlace al codigo", "https://github.com/dbjesuss/Salirdelpaso", True),
    ("Identificador persistente (DOI)", "pendiente: release de GitHub archivada en Zenodo (ver 10.7)", False),
    ("Tiempo de computo", "tabla 10.5; anexo A1 (E3)", True),
], columns=["requisito", "donde", "cumple"]).set_index("requisito")
print(f"Cumplidos: {int(lista['cumple'].sum())} de {len(lista)}")
lista

Cumplidos: 15 de 16


,donde,cumple
requisito,,
Versiones del software,tabla 10.1; requirements.txt; requirements_e4.txt,True
Hardware,tabla 10.1 y sec. 10.1,True
Semillas,tabla 10.2,True
Rangos de hiperparametros,tablas 10.3 y 10.3b; src/espacios.py; experiments/config_e4.py,True
Numero de ejecuciones y dispersion,tabla 10.5; tabla maestra,True
Tabla maestra (140 comb. + original + conjuntos adicionales),resultados_e4/tabla_maestra_e4.csv|parquet,True
Protocolo pre-registrado con tag de Git,experiments/protocolo.md; tags protocolo-e4 y protocolo-v2-evaluacion-ampliada,True
Codigo que genera cada resultado,experiments/ (scripts e9-e15) y CORRER_*.bat,True
Pruebas unitarias en integracion continua,.github/workflows/pytest.yml; 45 pruebas en tests/,True


## 10.7 Código, datos e identificador persistente

- **Código:** <https://github.com/dbjesuss/Salirdelpaso>.
- **Datos:** competencia *Home Credit Default Risk* {cite}`montoya2018homecredit`. Por las reglas de la competencia los datos no se
  redistribuyen: se descargan de Kaggle y se procesan con el capítulo 01 del anexo. Los conjuntos adicionales se descargan
  automáticamente de OpenML {cite}`vanschoren2014openml` (identificadores 31, 42477 y 1461).
- **DOI:** pendiente. Se obtiene creando una *release* del repositorio en GitHub con la integración de Zenodo activada. El DOI que
  asigne Zenodo se cita aquí y en `CITATION.cff`.

## 10.8 Bibliografía

Todas las referencias del libro están en `references.bib` (formato BibTeX). La lista siguiente reúne las que se citan en
cualquier capítulo del Entregable 4; cada cita del texto enlaza aquí.

```{bibliography}
```